In [ ]:
import os
import numpy as np
import rasterio
import cv2
from tqdm import tqdm

In [8]:
B2_PATH = r"F:\work\python\image frequency\landsat\LC09_L2SP_014016_20260325_20260326_02_T1_SR_B2.TIF"
B3_PATH = r"F:\work\python\image frequency\landsat\LC09_L2SP_014016_20260325_20260326_02_T1_SR_B3.TIF"
B4_PATH = r"F:\work\python\image frequency\landsat\LC09_L2SP_014016_20260325_20260326_02_T1_SR_B5.TIF"

OUTPUT_DIR = r"F:\work\python\image frequency\landsat\rgb_patches_256"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [9]:
def load_band(path):
    with rasterio.open(path) as src:
        return src.read(1)

b2 = load_band(B2_PATH)
b3 = load_band(B3_PATH)
b4 = load_band(B4_PATH)

print("Band shape:", b2.shape)

Band shape: (8371, 8321)


In [10]:
assert b2.shape == b3.shape == b4.shape, "Band size mismatch!"

In [11]:
# Stack → RGB (correct order)
image = np.stack([b4, b3, b2], axis=-1)

In [12]:
image = image.astype(np.float32)

# Normalize per channel
for i in range(3):
    band = image[:,:,i]
    band = (band - band.min()) / (band.max() - band.min() + 1e-8)
    image[:,:,i] = band

In [13]:
PATCH_SIZE = 256 
STRIDE = 64   # overlap → better performance

count = 0

for i in tqdm(range(0, image.shape[0] - PATCH_SIZE, STRIDE)):
    for j in range(0, image.shape[1] - PATCH_SIZE, STRIDE):

        patch = image[i:i+PATCH_SIZE, j:j+PATCH_SIZE]

        # Skip very dark / empty patches
        if np.mean(patch) < 0.05:
            continue

        # Convert to 0–255
        patch = (patch * 255).astype("uint8")

        save_path = os.path.join(
            OUTPUT_DIR,
            f"patch_{count}.png"
        )

        cv2.imwrite(save_path, patch)
        count += 1

print("Total RGB patches:", count)

100%|██████████| 127/127 [00:29<00:00,  4.35it/s]

Total RGB patches: 10575


In [14]:
import os
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.cluster import KMeans

In [15]:
INPUT_DIR = r"F:\work\python\image frequency\landsat\rgb_patches_256"
OUTPUT_DIR = r"F:\work\python\image frequency\landsat\\categorized_patches"

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [16]:
def extract_features(img):

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # 🔹 Intensity
    intensity = np.mean(gray) / 255.0

    # 🔹 Texture (std deviation)
    std = np.std(gray) / 255.0

    # 🔹 Edge density
    edges = cv2.Canny(gray, 100, 200)
    edge_density = np.sum(edges > 0) / edges.size

    return [intensity, std, edge_density]

In [17]:
data = []
file_list = os.listdir(INPUT_DIR)

for fname in tqdm(file_list):

    path = os.path.join(INPUT_DIR, fname)
    img = cv2.imread(path)

    if img is None:
        continue

    features = extract_features(img)

    data.append({
        "filename": fname,
        "intensity": features[0],
        "std": features[1],
        "edge": features[2]
    })

df = pd.DataFrame(data)
print(df.head())

100%|██████████| 10575/10575 [03:25<00:00, 51.46it/s]


         filename  intensity       std      edge
0     patch_0.png   0.067813  0.213482  0.007690
1     patch_1.png   0.212483  0.334108  0.028915
2    patch_10.png   0.104504  0.258467  0.021515
3   patch_100.png   0.687393  0.128420  0.099594
4  patch_1000.png   0.616451  0.182113  0.111176


In [18]:
kmeans = KMeans(n_clusters=3, random_state=42)
df["cluster"] = kmeans.fit_predict(df[["intensity","std","edge"]])

In [19]:
# Sort clusters by complexity (using std + edge)
cluster_score = df.groupby("cluster")[["std","edge"]].mean().sum(axis=1)

sorted_clusters = cluster_score.sort_values().index.tolist()

mapping = {
    sorted_clusters[0]: "low",
    sorted_clusters[1]: "medium",
    sorted_clusters[2]: "high"
}

df["category"] = df["cluster"].map(mapping)

print(df["category"].value_counts())

category
low       5811
high      3370
medium    1394
Name: count, dtype: int64


In [20]:
for cat in ["low","medium","high"]:
    os.makedirs(os.path.join(OUTPUT_DIR, cat), exist_ok=True)

for _, row in tqdm(df.iterrows(), total=len(df)):

    src = os.path.join(INPUT_DIR, row["filename"])
    dst = os.path.join(OUTPUT_DIR, row["category"], row["filename"])

    img = cv2.imread(src)

    if img is not None:
        cv2.imwrite(dst, img)

print("Categorization completed!")

100%|██████████| 10575/10575 [01:52<00:00, 93.85it/s]

Categorization completed!
